In [ ]:
from pathlib import Path
from awpy import Demo

demos = sorted(Path("../../data/demos").glob("*.dem"))
demos

In [ ]:
import polars as pl
from awpy import Demo

dem = Demo(demos[0])
dem.parse(events=[*dem.default_events, "player_blind"])

kills, damages, rounds = dem.kills, dem.damages, dem.rounds

NAME = "WilloKage"
me = dem.ticks.filter(pl.col("name") == NAME)["steamid"][0]

# Équipe de départ de chaque joueur (CT = 0, T = 1), comme à l'ingestion
start_side = (
    dem.ticks.filter(pl.col("steamid").is_not_null() & pl.col("side").is_not_null())
    .sort("tick")
    .group_by("steamid")
    .agg(pl.col("side").first().str.to_lowercase().alias("s"))
)
team = {sid: (0 if s == "ct" else 1 if s == "t" else None) for sid, s in start_side.iter_rows()}
my_team = team[me]
print(NAME, me, "équipe", my_team)

In [ ]:
sorted(damages["weapon"].unique())

In [ ]:
print(rounds.columns)
t0 = rounds["freeze_end"].min()
t1 = rounds["official_end"].max()

def in_game(df):
    return df.filter((pl.col("tick") >= t0) & (pl.col("tick") <= t1))

In [ ]:
mine = damages.filter(
    (pl.col("attacker_steamid") == me)
    & (pl.col("attacker_side") != pl.col("victim_side"))
)

print("HE :", mine.filter(pl.col("weapon") == "hegrenade")["dmg_health_real"].sum())
print("Molotov :", mine.filter(pl.col("weapon").is_in(["inferno", "molotov", "incgrenade"]))["dmg_health_real"].sum())

# Détail pour vérifier à l'oeil, round par round
(
    mine.filter(pl.col("weapon").is_in(["hegrenade", "inferno", "molotov", "incgrenade"]))
    .group_by("round_num", "weapon").agg(pl.col("dmg_health_real").sum())
    .sort("round_num")
)

In [ ]:
fa = kills.filter(
    (pl.col("assister_steamid") == me)
    & pl.col("assistedflash")
    & (pl.col("assister_side") != pl.col("victim_side"))
)
print("Flash assists :", fa.height)
fa.select("round_num", "victim_name", "tick")

In [ ]:
det = in_game(dem.events["flashbang_detonate"]).filter(pl.col("user_steamid") == me)
print("Flashs explosés :", det.height)

wf = in_game(dem.events["weapon_fire"])
print(wf["weapon"].unique().to_list())  # repère le nom exact du flash
fired = wf.filter((pl.col("user_steamid") == me) & pl.col("weapon").str.contains("flashbang"))
print("Flashs lancés :", fired.height)

In [ ]:
b = in_game(dem.events["player_blind"]).filter(pl.col("attacker_steamid") == me)

b = b.with_columns(
    pl.col("user_steamid").replace_strict(team, default=None).alias("victim_team")
)

enemy = b.filter(pl.col("victim_team") != my_team)
mates = b.filter((pl.col("victim_team") == my_team) & (pl.col("user_steamid") != me))

print("Ennemis aveuglés :", enemy.height, "| durée totale :", round(enemy["blind_duration"].sum(), 2), "s")
print("Alliés aveuglés :", mates.height)
print("Ennemis aveuglés plus de 1 s :", enemy.filter(pl.col("blind_duration") >= 1.0).height)

# Détail ligne par ligne
enemy.select("tick", "user_name", "blind_duration").sort("tick") if "user_name" in enemy.columns \
    else enemy.sort("tick")

In [ ]:
pl.DataFrame({
    "flashes_thrown": [det.height],
    "enemies_flashed": [enemy.height],
    "team_flashed": [mates.height],
    "flash_assists": [fa.height],
    "blind_time": [round(enemy["blind_duration"].sum(), 2)],
    "he_damage": [mine.filter(pl.col("weapon") == "hegrenade")["dmg_health_real"].sum()],
    "molotov_damage": [mine.filter(pl.col("weapon").is_in(["inferno", "molotov", "incgrenade"]))["dmg_health_real"].sum()],
})

In [ ]:
j = kills.join(damages, on=["tick", "attacker_steamid", "victim_steamid"], suffix="_hurt")
j.group_by("weapon", "weapon_hurt").len().sort("weapon")